In [1]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Find our best model from the experiments
experiment = mlflow.get_experiment_by_name("fraud-detection")
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.auc_pr DESC"]
)

best_run = runs.iloc[0]
print(f"Best model: {best_run['tags.mlflow.runName']}")
print(f"AUC-PR: {best_run['metrics.auc_pr']:.4f}")
print(f"Run ID: {best_run['run_id']}")

Best model: deeper_xgboost
AUC-PR: 0.8040
Run ID: 6cd8950f27f844a2a5b1b85377c750e7


In [2]:
# Register the best model in MLflow Model Registry
model_name = "fraud-detection-model"

# Register the model from the best run
model_uri = f"runs:/{best_run['run_id']}/model"
result = mlflow.register_model(model_uri, model_name)

print(f"Model registered: {result.name}")
print(f"Version: {result.version}")
print(f"Source run: {best_run['tags.mlflow.runName']}")
print(f"\nThis model is now in the registry with version tracking.")
print(f"Think of this as 'git tag' but for ML models.")

Successfully registered model 'fraud-detection-model'.
2026/03/07 11:45:29 WARNING mlflow.tracking._model_registry.fluent: Run with id 6cd8950f27f844a2a5b1b85377c750e7 has no artifacts at artifact path 'model', registering model based on models:/m-979658b6be5d45b9a2ff9c0cc5d4044b instead


Model registered: fraud-detection-model
Version: 1
Source run: deeper_xgboost

This model is now in the registry with version tracking.
Think of this as 'git tag' but for ML models.


Created version '1' of model 'fraud-detection-model'.


In [3]:
# Add description and tags — this is what makes the registry auditable
# In production, a regulator asks "what is this model and who trained it?"

client.update_model_version(
    name=model_name,
    version=1,
    description=(
        "XGBoost fraud detection model trained on 227,845 transactions. "
        "300 trees, max depth 8, learning rate 0.05. "
        "AUC-PR: 0.804, Precision: 87.7%, Recall: 76.0%. "
        "Catches 57/75 frauds with only 8 false alarms."
    )
)

# Tag with metadata
client.set_model_version_tag(model_name, 1, "training_dataset", "kaggle_creditcard_v1")
client.set_model_version_tag(model_name, 1, "trained_by", "ltiwari")
client.set_model_version_tag(model_name, 1, "use_case", "transaction_fraud_detection")
client.set_model_version_tag(model_name, 1, "regulatory_review", "pending")

# Transition to Staging — model needs validation before production
client.transition_model_version_stage(
    name=model_name,
    version=1,
    stage="Staging"
)

print(f"Model: {model_name}")
print(f"Version: 1")
print(f"Stage: Staging")
print(f"\nLifecycle stages:")
print(f"  None → Staging → Production → Archived")
print(f"           ↑")
print(f"       YOU ARE HERE")
print(f"\nModel metadata tags:")
for tag in client.get_model_version(model_name, 1).tags:
    print(f"  {tag}: {client.get_model_version(model_name, 1).tags[tag]}")

print(f"\nThis model CANNOT go to Production until it passes governance checks.")
print(f"That's Build 2 — bias detection, explainability, audit trail.")

Model: fraud-detection-model
Version: 1
Stage: Staging

Lifecycle stages:
  None → Staging → Production → Archived
           ↑
       YOU ARE HERE

Model metadata tags:
  training_dataset: kaggle_creditcard_v1
  trained_by: ltiwari
  use_case: transaction_fraud_detection
  regulatory_review: pending

This model CANNOT go to Production until it passes governance checks.
That's Build 2 — bias detection, explainability, audit trail.


/var/folders/mq/vrcy8r_12fl7xdnmnj12cxbm0000gn/T/ipykernel_59911/930034462.py:22: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [4]:
# CHAMPION vs CHALLENGER — the production pattern
# Register the baseline model as Version 2 to demonstrate versioning

baseline_run = runs[runs['tags.mlflow.runName'] == 'baseline_xgboost'].iloc[0]
baseline_uri = f"runs:/{baseline_run['run_id']}/model"
result2 = mlflow.register_model(baseline_uri, model_name)

client.update_model_version(
    name=model_name,
    version=2,
    description=(
        "Baseline XGBoost model. 100 trees, max depth 6, learning rate 0.1. "
        "AUC-PR: 0.773, Precision: 80.3%, Recall: 76.0%. "
        "14 false alarms — more than Version 1."
    )
)

# Now compare them side by side
print("CHAMPION vs CHALLENGER COMPARISON")
print("=" * 60)
print(f"\n{'Metric':<20} {'v1 (Staging)':<18} {'v2 (Challenger)'}")
print("-" * 55)

v1_run = runs[runs['tags.mlflow.runName'] == 'deeper_xgboost'].iloc[0]
v2_run = baseline_run

metrics = ['auc_pr', 'precision', 'recall', 'false_positives', 'false_negatives']
labels = ['AUC-PR', 'Precision', 'Recall', 'False Alarms', 'Missed Frauds']

for label, metric in zip(labels, metrics):
    v1_val = v1_run[f'metrics.{metric}']
    v2_val = v2_run[f'metrics.{metric}']
    winner = "←" if v1_val >= v2_val else "→"
    if metric in ['false_positives', 'false_negatives']:
        winner = "←" if v1_val <= v2_val else "→"
    print(f"{label:<20} {v1_val:<18.4f} {v2_val:.4f}  {winner} better")

# Show all versions in registry
print(f"\n\nMODEL REGISTRY — All Versions")
print("=" * 60)
for v in client.search_model_versions(f"name='{model_name}'"):
    print(f"  Version {v.version}: Stage={v.current_stage:<12} Run={v.description[:50]}...")

print(f"\nKey concept: In production, Version 1 (champion) serves traffic.")
print(f"Version 2 (challenger) is tested against it. Only if the challenger")
print(f"wins on key metrics does it get promoted. This is automated in Phase 1F.")

Registered model 'fraud-detection-model' already exists. Creating a new version of this model...
2026/03/07 11:48:02 WARNING mlflow.tracking._model_registry.fluent: Run with id 34aab2611e4842d8bcc0cb3c5ccbbcaa has no artifacts at artifact path 'model', registering model based on models:/m-21161baf028140dba632a138b64ec12a instead


CHAMPION vs CHALLENGER COMPARISON

Metric               v1 (Staging)       v2 (Challenger)
-------------------------------------------------------
AUC-PR               0.8040             0.7734  ← better
Precision            0.8769             0.8028  ← better
Recall               0.7600             0.7600  ← better
False Alarms         8.0000             14.0000  ← better
Missed Frauds        18.0000            18.0000  ← better


MODEL REGISTRY — All Versions
  Version 2: Stage=None         Run=Baseline XGBoost model. 100 trees, max depth 6, le...
  Version 1: Stage=Staging      Run=XGBoost fraud detection model trained on 227,845 t...

Key concept: In production, Version 1 (champion) serves traffic.
Version 2 (challenger) is tested against it. Only if the challenger
wins on key metrics does it get promoted. This is automated in Phase 1F.


Created version '2' of model 'fraud-detection-model'.


In [5]:
%%writefile pipeline/registry/model_registry.py
"""
Model Registry Management for Fraud Detection
Enterprise MLOps Platform

Handles model registration, versioning, lifecycle transitions,
and champion-challenger comparisons.

Lifecycle: None → Staging → Production → Archived
A model CANNOT reach Production without passing governance gates.
"""

import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()
MODEL_NAME = "fraud-detection-model"


def register_model(run_id: str, description: str, trained_by: str = "system"):
    """Register a trained model from an MLflow run."""
    model_uri = f"runs:/{run_id}/model"
    result = mlflow.register_model(model_uri, MODEL_NAME)
    
    client.update_model_version(
        name=MODEL_NAME,
        version=result.version,
        description=description
    )
    client.set_model_version_tag(MODEL_NAME, result.version, "trained_by", trained_by)
    client.set_model_version_tag(MODEL_NAME, result.version, "regulatory_review", "pending")
    
    print(f"Registered {MODEL_NAME} v{result.version}")
    return result.version


def promote_to_staging(version: int):
    """Move a model version to Staging for validation."""
    client.transition_model_version_stage(MODEL_NAME, version, "Staging")
    print(f"v{version} → Staging")


def promote_to_production(version: int):
    """
    Move a model version to Production.
    In a governed pipeline, this should ONLY be called after
    all governance gates pass (bias, explainability, audit).
    """
    # Archive current production model
    for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
        if v.current_stage == "Production":
            client.transition_model_version_stage(MODEL_NAME, int(v.version), "Archived")
            print(f"v{v.version} → Archived (was Production)")
    
    client.transition_model_version_stage(MODEL_NAME, version, "Production")
    print(f"v{version} → Production")


def rollback():
    """
    Emergency rollback — revert to the most recent archived model.
    This is the 'oh no' button when a production model degrades.
    """
    archived = [
        v for v in client.search_model_versions(f"name='{MODEL_NAME}'")
        if v.current_stage == "Archived"
    ]
    if not archived:
        print("No archived model to rollback to.")
        return
    
    latest_archived = sorted(archived, key=lambda v: int(v.version), reverse=True)[0]
    
    # Demote current production
    for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
        if v.current_stage == "Production":
            client.transition_model_version_stage(MODEL_NAME, int(v.version), "Archived")
    
    # Restore archived version
    client.transition_model_version_stage(MODEL_NAME, int(latest_archived.version), "Production")
    print(f"Rolled back to v{latest_archived.version}")


def get_production_model():
    """Load the current production model for serving."""
    for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
        if v.current_stage == "Production":
            model = mlflow.xgboost.load_model(f"models:/{MODEL_NAME}/{v.version}")
            print(f"Loaded production model v{v.version}")
            return model, int(v.version)
    print("No production model found.")
    return None, None


def compare_versions(v1: int, v2: int):
    """Compare two model versions side by side."""
    ver1 = client.get_model_version(MODEL_NAME, v1)
    ver2 = client.get_model_version(MODEL_NAME, v2)
    
    run1 = client.get_run(ver1.run_id)
    run2 = client.get_run(ver2.run_id)
    
    print(f"\nv{v1} ({ver1.current_stage}) vs v{v2} ({ver2.current_stage})")
    print("-" * 50)
    
    for metric in ["auc_pr", "precision", "recall"]:
        val1 = run1.data.metrics.get(metric, 0)
        val2 = run2.data.metrics.get(metric, 0)
        winner = f"v{v1}" if val1 >= val2 else f"v{v2}"
        print(f"  {metric:<15} v{v1}: {val1:.4f}  v{v2}: {val2:.4f}  → {winner}")


if __name__ == "__main__":
    print(f"Registry: {MODEL_NAME}")
    print(f"\nAll versions:")
    for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
        print(f"  v{v.version}: {v.current_stage}")

Writing pipeline/registry/model_registry.py


In [6]:
%%writefile pipeline/serving/app.py
"""
Model Serving API for Fraud Detection
Enterprise MLOps Platform

FastAPI REST endpoint that loads the production model from the registry
and serves real-time fraud predictions with explanations.

Think of this as Express.js but for Python — you'll feel at home.
"""

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import numpy as np
import mlflow
import mlflow.xgboost
from mlflow.tracking import MlflowClient
from datetime import datetime
import json
import os

app = FastAPI(
    title="Fraud Detection API",
    description="Real-time fraud scoring for transaction data",
    version="1.0.0"
)

# ============================================================
# REQUEST / RESPONSE SCHEMAS (like TypeScript interfaces)
# ============================================================

class Transaction(BaseModel):
    """Single transaction to score."""
    V1: float = Field(..., description="PCA feature V1")
    V2: float = Field(..., description="PCA feature V2")
    V3: float = Field(..., description="PCA feature V3")
    V4: float = Field(..., description="PCA feature V4")
    V5: float = Field(..., description="PCA feature V5")
    V6: float = Field(..., description="PCA feature V6")
    V7: float = Field(..., description="PCA feature V7")
    V8: float = Field(..., description="PCA feature V8")
    V9: float = Field(..., description="PCA feature V9")
    V10: float = Field(..., description="PCA feature V10")
    V11: float = Field(..., description="PCA feature V11")
    V12: float = Field(..., description="PCA feature V12")
    V13: float = Field(..., description="PCA feature V13")
    V14: float = Field(..., description="PCA feature V14")
    V15: float = Field(..., description="PCA feature V15")
    V16: float = Field(..., description="PCA feature V16")
    V17: float = Field(..., description="PCA feature V17")
    V18: float = Field(..., description="PCA feature V18")
    V19: float = Field(..., description="PCA feature V19")
    V20: float = Field(..., description="PCA feature V20")
    V21: float = Field(..., description="PCA feature V21")
    V22: float = Field(..., description="PCA feature V22")
    V23: float = Field(..., description="PCA feature V23")
    V24: float = Field(..., description="PCA feature V24")
    V25: float = Field(..., description="PCA feature V25")
    V26: float = Field(..., description="PCA feature V26")
    V27: float = Field(..., description="PCA feature V27")
    V28: float = Field(..., description="PCA feature V28")
    Amount: float = Field(..., description="Transaction amount in dollars")
    Time: float = Field(default=0.0, description="Seconds from first transaction")

class PredictionResponse(BaseModel):
    """Fraud prediction result."""
    fraud_probability: float
    risk_tier: str
    is_fraud: bool
    model_version: int
    prediction_time_ms: float
    top_features: List[dict]

class HealthResponse(BaseModel):
    """API health status."""
    status: str
    model_loaded: bool
    model_version: Optional[int]
    uptime_seconds: float
    total_predictions: int

# ============================================================
# MODEL LOADING
# ============================================================

MODEL_NAME = "fraud-detection-model"
model = None
model_version = None
start_time = datetime.now()
prediction_count = 0
prediction_log = []

def load_model():
    """Load the staging model from registry (production once governance passes)."""
    global model, model_version
    client = MlflowClient()
    
    # Try Production first, fall back to Staging
    for stage in ["Production", "Staging"]:
        versions = [
            v for v in client.search_model_versions(f"name='{MODEL_NAME}'")
            if v.current_stage == stage
        ]
        if versions:
            latest = sorted(versions, key=lambda v: int(v.version), reverse=True)[0]
            model = mlflow.xgboost.load_model(f"models:/{MODEL_NAME}/{latest.version}")
            model_version = int(latest.version)
            print(f"Loaded {MODEL_NAME} v{model_version} ({stage})")
            return
    
    print("WARNING: No model found in registry")

# ============================================================
# FEATURE ENGINEERING (inline for serving — same logic as training)
# ============================================================

def engineer_serving_features(tx: dict) -> np.ndarray:
    """
    Apply the same feature engineering used in training.
    CRITICAL: This MUST match pipeline/ingestion/feature_engineering.py
    Any difference = training-serving skew = silent model degradation.
    """
    amount = tx['Amount']
    time_val = tx['Time']
    
    # Time features
    hour = (time_val / 3600) % 24
    is_night = 1 if 0 <= hour < 6 else 0
    
    # Amount features
    amount_log = np.log1p(amount)
    amount_zscore = (amount - 88.29) / 250.11  # training set mean/std
    
    if amount <= 1: amount_bin = 0
    elif amount <= 10: amount_bin = 1
    elif amount <= 50: amount_bin = 2
    elif amount <= 200: amount_bin = 3
    elif amount <= 1000: amount_bin = 4
    else: amount_bin = 5
    
    is_round = 1 if amount % 10 == 0 else 0
    
    # Interaction features
    v14_x_v12 = tx['V14'] * tx['V12']
    v17_x_amount = tx['V17'] * amount_log
    v14_x_amount = tx['V14'] * amount_log
    v14_v11_ratio = tx['V14'] / (tx['V11'] + 1e-6)
    
    # Risk signal
    fraud_risk_signal = -(tx['V17'] + tx['V14'] + tx['V12'] + tx['V10'])
    
    # Build feature array in same order as training
    raw_features = [tx[f'V{i}'] for i in range(1, 29)]
    raw_features.append(amount)
    engineered = [hour, is_night, amount_log, amount_zscore, amount_bin,
                  is_round, v14_x_v12, v17_x_amount, v14_x_amount,
                  v14_v11_ratio, fraud_risk_signal]
    
    return np.array(raw_features + engineered).reshape(1, -1)

# ============================================================
# ENDPOINTS
# ============================================================

@app.on_event("startup")
def startup():
    load_model()

@app.get("/health", response_model=HealthResponse)
def health_check():
    """Health check — is the model loaded and ready?"""
    uptime = (datetime.now() - start_time).total_seconds()
    return HealthResponse(
        status="healthy" if model is not None else "degraded",
        model_loaded=model is not None,
        model_version=model_version,
        uptime_seconds=round(uptime, 2),
        total_predictions=prediction_count
    )

@app.post("/predict", response_model=PredictionResponse)
def predict(transaction: Transaction):
    """Score a single transaction for fraud."""
    global prediction_count
    
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    pred_start = datetime.now()
    
    # Engineer features
    tx_dict = transaction.dict()
    features = engineer_serving_features(tx_dict)
    
    # Predict
    fraud_prob = float(model.predict_proba(features)[0][1])
    is_fraud = fraud_prob >= 0.5
    
    # Risk tier
    if fraud_prob >= 0.8: risk_tier = "critical"
    elif fraud_prob >= 0.5: risk_tier = "high"
    elif fraud_prob >= 0.3: risk_tier = "medium"
    else: risk_tier = "low"
    
    # Feature importance for this prediction
    feature_names = [f'V{i}' for i in range(1, 29)] + ['Amount'] + [
        'hour', 'is_night', 'amount_log', 'amount_zscore', 'amount_bin',
        'is_round_amount', 'v14_x_v12', 'v17_x_amount', 'v14_x_amount',
        'v14_v11_ratio', 'fraud_risk_signal'
    ]
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[-5:][::-1]
    top_features = [
        {"feature": feature_names[i], "importance": round(float(importances[i]), 4)}
        for i in top_idx
    ]
    
    pred_time = (datetime.now() - pred_start).total_seconds() * 1000
    
    # Log prediction
    prediction_count += 1
    prediction_log.append({
        "timestamp": datetime.now().isoformat(),
        "fraud_probability": fraud_prob,
        "risk_tier": risk_tier,
        "model_version": model_version,
        "latency_ms": round(pred_time, 2)
    })
    
    return PredictionResponse(
        fraud_probability=round(fraud_prob, 6),
        risk_tier=risk_tier,
        is_fraud=is_fraud,
        model_version=model_version,
        prediction_time_ms=round(pred_time, 2),
        top_features=top_features
    )

@app.get("/predictions/recent")
def recent_predictions():
    """Return recent prediction log — for monitoring."""
    return {"predictions": prediction_log[-20:], "total": prediction_count}

Writing pipeline/serving/app.py


In [8]:
import numpy as np
import pandas as pd
import pipeline.serving.app as serving

# Load the model
serving.load_model()

# Grab a real fraud and legitimate transaction
df = pd.read_csv('data/raw/creditcard.csv')
fraud_tx = df[df['Class'] == 1].iloc[0]
legit_tx = df[df['Class'] == 0].iloc[0]

# Score the fraud transaction
fraud_dict = {f'V{i}': fraud_tx[f'V{i}'] for i in range(1, 29)}
fraud_dict['Amount'] = fraud_tx['Amount']
fraud_dict['Time'] = fraud_tx['Time']
fraud_features = serving.engineer_serving_features(fraud_dict)
fraud_prob = float(serving.model.predict_proba(fraud_features)[0][1])

# Score the legitimate transaction
legit_dict = {f'V{i}': legit_tx[f'V{i}'] for i in range(1, 29)}
legit_dict['Amount'] = legit_tx['Amount']
legit_dict['Time'] = legit_tx['Time']
legit_features = serving.engineer_serving_features(legit_dict)
legit_prob = float(serving.model.predict_proba(legit_features)[0][1])

print("FRAUD TRANSACTION")
print(f"  Amount: ${fraud_tx['Amount']:.2f}")
print(f"  Fraud probability: {fraud_prob:.6f}")
print(f"  Risk tier: {'critical' if fraud_prob >= 0.8 else 'high' if fraud_prob >= 0.5 else 'medium' if fraud_prob >= 0.3 else 'low'}")
print(f"  Verdict: {'FRAUD DETECTED' if fraud_prob >= 0.5 else 'Cleared'}")

print(f"\nLEGITIMATE TRANSACTION")
print(f"  Amount: ${legit_tx['Amount']:.2f}")
print(f"  Fraud probability: {legit_prob:.6f}")
print(f"  Risk tier: {'critical' if legit_prob >= 0.8 else 'high' if legit_prob >= 0.5 else 'medium' if legit_prob >= 0.3 else 'low'}")
print(f"  Verdict: {'FRAUD DETECTED' if legit_prob >= 0.5 else 'Cleared'}")

print(f"\nThe model correctly separated fraud from legitimate.")
print(f"This is exactly what would happen at Mastercard — 5,000 times per second.")

Loaded fraud-detection-model v1 (Staging)
FRAUD TRANSACTION
  Amount: $0.00
  Fraud probability: 0.999795
  Risk tier: critical
  Verdict: FRAUD DETECTED

LEGITIMATE TRANSACTION
  Amount: $149.62
  Fraud probability: 0.000086
  Risk tier: low
  Verdict: Cleared

The model correctly separated fraud from legitimate.
This is exactly what would happen at Mastercard — 5,000 times per second.


In [9]:
%%writefile pipeline/serving/app.py
"""
Model Serving API for Fraud Detection
Enterprise MLOps Platform
"""

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import numpy as np
import mlflow
import mlflow.xgboost
from mlflow.tracking import MlflowClient
from datetime import datetime

app = FastAPI(
    title="Fraud Detection API",
    description="Real-time fraud scoring for transaction data",
    version="1.0.0"
)

# Schemas
class Transaction(BaseModel):
    V1: float; V2: float; V3: float; V4: float; V5: float
    V6: float; V7: float; V8: float; V9: float; V10: float
    V11: float; V12: float; V13: float; V14: float; V15: float
    V16: float; V17: float; V18: float; V19: float; V20: float
    V21: float; V22: float; V23: float; V24: float; V25: float
    V26: float; V27: float; V28: float
    Amount: float
    Time: float = 0.0

class PredictionResponse(BaseModel):
    fraud_probability: float
    risk_tier: str
    is_fraud: bool
    model_version: int
    prediction_time_ms: float
    top_features: List[dict]

class HealthResponse(BaseModel):
    status: str
    model_loaded: bool
    model_version: Optional[int]
    uptime_seconds: float
    total_predictions: int

# State
MODEL_NAME = "fraud-detection-model"
model = None
model_version = None
start_time = datetime.now()
prediction_count = 0
prediction_log = []


def load_model():
    global model, model_version
    client = MlflowClient()
    
    for stage in ["Production", "Staging"]:
        versions = [
            v for v in client.search_model_versions(f"name='{MODEL_NAME}'")
            if v.current_stage == stage
        ]
        if versions:
            latest = sorted(versions, key=lambda v: int(v.version), reverse=True)[0]
            model = mlflow.xgboost.load_model(f"models:/{MODEL_NAME}/{latest.version}")
            model_version = int(latest.version)
            print(f"Loaded {MODEL_NAME} v{model_version} ({stage})")
            return
    
    print("WARNING: No model found in registry")


def engineer_serving_features(tx: dict) -> np.ndarray:
    """Same feature logic as training pipeline — prevents training-serving skew."""
    amount = tx['Amount']
    time_val = tx['Time']
    
    hour = (time_val / 3600) % 24
    is_night = 1 if 0 <= hour < 6 else 0
    amount_log = np.log1p(amount)
    amount_zscore = (amount - 88.29) / 250.11
    
    if amount <= 1: amount_bin = 0
    elif amount <= 10: amount_bin = 1
    elif amount <= 50: amount_bin = 2
    elif amount <= 200: amount_bin = 3
    elif amount <= 1000: amount_bin = 4
    else: amount_bin = 5
    
    is_round = 1 if amount % 10 == 0 else 0
    v14_x_v12 = tx['V14'] * tx['V12']
    v17_x_amount = tx['V17'] * amount_log
    v14_x_amount = tx['V14'] * amount_log
    v14_v11_ratio = tx['V14'] / (tx['V11'] + 1e-6)
    fraud_risk_signal = -(tx['V17'] + tx['V14'] + tx['V12'] + tx['V10'])
    
    raw_features = [tx[f'V{i}'] for i in range(1, 29)]
    raw_features.append(amount)
    engineered = [hour, is_night, amount_log, amount_zscore, amount_bin,
                  is_round, v14_x_v12, v17_x_amount, v14_x_amount,
                  v14_v11_ratio, fraud_risk_signal]
    
    return np.array(raw_features + engineered).reshape(1, -1)


@app.on_event("startup")
def startup():
    load_model()


@app.get("/health", response_model=HealthResponse)
def health_check():
    uptime = (datetime.now() - start_time).total_seconds()
    return HealthResponse(
        status="healthy" if model is not None else "degraded",
        model_loaded=model is not None,
        model_version=model_version,
        uptime_seconds=round(uptime, 2),
        total_predictions=prediction_count
    )


@app.post("/predict", response_model=PredictionResponse)
def predict(transaction: Transaction):
    global prediction_count
    
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    pred_start = datetime.now()
    
    tx_dict = transaction.dict()
    features = engineer_serving_features(tx_dict)
    fraud_prob = float(model.predict_proba(features)[0][1])
    is_fraud = fraud_prob >= 0.5
    
    if fraud_prob >= 0.8: risk_tier = "critical"
    elif fraud_prob >= 0.5: risk_tier = "high"
    elif fraud_prob >= 0.3: risk_tier = "medium"
    else: risk_tier = "low"
    
    feature_names = [f'V{i}' for i in range(1, 29)] + ['Amount'] + [
        'hour', 'is_night', 'amount_log', 'amount_zscore', 'amount_bin',
        'is_round_amount', 'v14_x_v12', 'v17_x_amount', 'v14_x_amount',
        'v14_v11_ratio', 'fraud_risk_signal'
    ]
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[-5:][::-1]
    top_features = [
        {"feature": feature_names[i], "importance": round(float(importances[i]), 4)}
        for i in top_idx
    ]
    
    pred_time = (datetime.now() - pred_start).total_seconds() * 1000
    
    prediction_count += 1
    prediction_log.append({
        "timestamp": datetime.now().isoformat(),
        "fraud_probability": fraud_prob,
        "risk_tier": risk_tier,
        "model_version": model_version,
        "latency_ms": round(pred_time, 2)
    })
    
    return PredictionResponse(
        fraud_probability=round(fraud_prob, 6),
        risk_tier=risk_tier,
        is_fraud=is_fraud,
        model_version=model_version,
        prediction_time_ms=round(pred_time, 2),
        top_features=top_features
    )


@app.get("/predictions/recent")
def recent_predictions():
    return {"predictions": prediction_log[-20:], "total": prediction_count}

Overwriting pipeline/serving/app.py


In [10]:
%%writefile pipeline/registry/model_registry.py
"""
Model Registry Management
Enterprise MLOps Platform
"""

import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()
MODEL_NAME = "fraud-detection-model"


def register_model(run_id: str, description: str, trained_by: str = "system"):
    model_uri = f"runs:/{run_id}/model"
    result = mlflow.register_model(model_uri, MODEL_NAME)
    
    client.update_model_version(name=MODEL_NAME, version=result.version, description=description)
    client.set_model_version_tag(MODEL_NAME, result.version, "trained_by", trained_by)
    client.set_model_version_tag(MODEL_NAME, result.version, "regulatory_review", "pending")
    
    print(f"Registered {MODEL_NAME} v{result.version}")
    return result.version


def promote_to_staging(version: int):
    client.transition_model_version_stage(MODEL_NAME, version, "Staging")
    print(f"v{version} → Staging")


def promote_to_production(version: int):
    for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
        if v.current_stage == "Production":
            client.transition_model_version_stage(MODEL_NAME, int(v.version), "Archived")
            print(f"v{v.version} → Archived (was Production)")
    
    client.transition_model_version_stage(MODEL_NAME, version, "Production")
    print(f"v{version} → Production")


def rollback():
    archived = [
        v for v in client.search_model_versions(f"name='{MODEL_NAME}'")
        if v.current_stage == "Archived"
    ]
    if not archived:
        print("No archived model to rollback to.")
        return
    
    latest_archived = sorted(archived, key=lambda v: int(v.version), reverse=True)[0]
    
    for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
        if v.current_stage == "Production":
            client.transition_model_version_stage(MODEL_NAME, int(v.version), "Archived")
    
    client.transition_model_version_stage(MODEL_NAME, int(latest_archived.version), "Production")
    print(f"Rolled back to v{latest_archived.version}")


def get_production_model():
    for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
        if v.current_stage == "Production":
            model = mlflow.xgboost.load_model(f"models:/{MODEL_NAME}/{v.version}")
            print(f"Loaded production model v{v.version}")
            return model, int(v.version)
    print("No production model found.")
    return None, None


def compare_versions(v1: int, v2: int):
    ver1 = client.get_model_version(MODEL_NAME, v1)
    ver2 = client.get_model_version(MODEL_NAME, v2)
    run1 = client.get_run(ver1.run_id)
    run2 = client.get_run(ver2.run_id)
    
    print(f"\nv{v1} ({ver1.current_stage}) vs v{v2} ({ver2.current_stage})")
    print("-" * 50)
    for metric in ["auc_pr", "precision", "recall"]:
        val1 = run1.data.metrics.get(metric, 0)
        val2 = run2.data.metrics.get(metric, 0)
        winner = f"v{v1}" if val1 >= val2 else f"v{v2}"
        print(f"  {metric:<15} v{v1}: {val1:.4f}  v{v2}: {val2:.4f}  → {winner}")


if __name__ == "__main__":
    print(f"Registry: {MODEL_NAME}")
    for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
        print(f"  v{v.version}: {v.current_stage}")

Overwriting pipeline/registry/model_registry.py
